# PPO-Clipped with Gymnasium
This notebook demonstrates how to train an agent using PPO (Clipped version) with Gymnasium's CartPole environment.

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical
from collections import namedtuple
import matplotlib.pyplot as plt

In [2]:
env = gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

In [3]:
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, n_actions),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.fc(x)

class ValueNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.fc(x)

In [4]:
Transition = namedtuple('Transition', ['state', 'action', 'log_prob', 'reward', 'done', 'value'])

def compute_gae(transitions, gamma=0.99, lam=0.95):
    rewards = [t.reward for t in transitions]
    values = [t.value for t in transitions] + [0]
    dones = [t.done for t in transitions]

    gae = 0
    returns = []
    for step in reversed(range(len(transitions))):
        delta = rewards[step] + gamma * values[step + 1] * (1 - dones[step]) - values[step]
        gae = delta + gamma * lam * (1 - dones[step]) * gae
        returns.insert(0, gae + values[step])
    return returns